<a href="https://colab.research.google.com/github/JayasreeMeda/ps1/blob/main/ps1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ps1 - I/O + descriptive stats
**Jayasree**

 Small look at three public datasets in three different formats - how big countries are, how they compare on population, and what a US demographic sample looks like.

**Data**
1. **HTML** - Wikipedia, *List of countries by population (UN)* - `https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)`
2. **JSON** - World Bank population indicator (2022) - `https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&date=2022&per_page=400`
3. **SAS xpt** - CDC NHANES 2021-2023 Demographics - `https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt`


In [ ]:
import urllib.request, json, io
import pandas as pd

UA = {"User-Agent": "Mozilla/5.0"}


## 1. HTML - countries by population

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)"
req = urllib.request.Request(url, headers=UA)
html = urllib.request.urlopen(req).read().decode("utf-8")

pop = pd.read_html(io.StringIO(html))[0]
pop = pop.rename(columns={
    "Country or territory": "country",
    "Population (1 July 2023)": "population",
    "Change (%)": "change_pct",
    "UN continental region[1]": "region",
})[["country", "population", "change_pct", "region"]]
pop = pop.dropna(subset=["country", "population"])
pop = pop[pop["country"] != "World"].head(50).reset_index(drop=True)

# Change (%) comes in as "+0.88%" — strip and cast to float
pop["change_pct"] = pd.to_numeric(pop["change_pct"].str.replace("%","",regex=False).str.replace("−","-",regex=False), errors="coerce")
pop.head()


,country,population,change_pct,region
0,India,1.438070e+09,0.89,Asia
1,China[a],1.422585e+09,-0.18,Asia
2,United States,3.434773e+08,0.57,Americas
3,Indonesia,2.811901e+08,0.85,Asia
4,Pakistan,2.475045e+08,1.56,Asia


In [ ]:
# Descriptive stats
pop["population"].describe().round(0)


count    5.000000e+01
mean     1.400083e+08
std      2.750489e+08
min      3.116565e+07
25%      3.932203e+07
50%      5.741923e+07
75%      1.148023e+08
max      1.438070e+09
Name: population, dtype: float64

In [ ]:
# Subset: countries over 100 million
pop[pop["population"] > 100_000_000]


,country,population,change_pct,region
0,India,1.438070e+09,0.89,Asia
1,China[a],1.422585e+09,-0.18,Asia
2,United States,3.434773e+08,0.57,Americas
3,Indonesia,2.811901e+08,0.85,Asia
4,Pakistan,2.475045e+08,1.56,Asia
5,Nigeria,2.278829e+08,2.12,Africa
6,Brazil,2.111407e+08,0.40,Americas
7,Bangladesh,1.714670e+08,1.23,Asia
8,Russia,1.454405e+08,-0.10,Europe
9,Mexico,1.297398e+08,0.88,Americas


In [ ]:
# Regex filter: countries containing "Republic"
pop[pop["country"].str.contains(r"Republic", regex=True)]


,country,population,change_pct,region


In [ ]:
# Groupby/agg: mean population and % change by region
pop.groupby("region")[["population", "change_pct"]].agg(["count", "mean"]).round(2)


population               change_pct      
              count          mean      count  mean
region                                            
Africa           16  6.944874e+07         16  2.24
Americas          7  1.221946e+08          7  0.80
Asia             19  2.360451e+08         19  0.97
Europe            8  6.862715e+07          8 -0.71

_Interpretation:_ population is very right-skewed - a handful of huge countries (India, China, US) pull the mean far above the median. Grouping by region shows Asia dominates in both size and count in this top-50 slice.


## 2. JSON - World Bank population API

In [ ]:
url = ("https://api.worldbank.org/v2/country/all/indicator/"
       "SP.POP.TOTL?format=json&date=2022&per_page=400")
raw = urllib.request.urlopen(url).read()
records = json.loads(raw)[1]

rc = pd.json_normalize(records)[["country.value", "countryiso3code", "value"]]
rc.columns = ["country", "iso3", "population"]
rc = rc.dropna(subset=["population"])
rc = rc[rc["iso3"].str.len() == 3]   # drop regional aggregates
rc.head()


,country,iso3,population
0,Africa Eastern and Southern,AFE,731821393.0
1,Africa Western and Central,AFW,496366058.0
2,Arab World,ARB,471352066.0
3,Caribbean small states,CSS,4497310.0
4,Central Europe and the Baltics,CEB,100071871.0


In [ ]:
# Descriptive stats
rc["population"].describe().round(0)


count    2.600000e+02
mean     2.985121e+08
std      9.882931e+08
min      9.992000e+03
25%      1.707245e+06
50%      1.043607e+07
75%      5.388071e+07
max      7.988550e+09
Name: population, dtype: float64

In [ ]:
# Subset: 10 most populous countries
rc.nlargest(10, "population")


,country,iso3,population
47,World,WLD,7.988550e+09
17,IDA & IBRD total,IBT,6.793703e+09
26,Low & middle income,LMY,6.590522e+09
32,Middle income,MIC,5.882746e+09
16,IBRD only,IBD,4.928196e+09
5,Early-demographic dividend,EAR,3.470645e+09
6,East Asia & Pacific,EAS,2.379617e+09
21,Late-demographic dividend,LTE,2.323800e+09
7,East Asia & Pacific (excluding high income),EAP,2.133560e+09
8,East Asia & Pacific (IDA & IBRD countries),TEA,2.107261e+09


In [ ]:
# Regex filter: countries starting with "United"
rc[rc["country"].str.contains(r"^United", regex=True)]


,country,iso3,population
252,United Arab Emirates,ARE,10074977.0
253,United Kingdom,GBR,67636000.0
254,United States,USA,333996304.0


In [ ]:
# Groupby/agg: quartile buckets by population
rc["bucket"] = pd.qcut(rc["population"], 4, labels=["Q1_small", "Q2", "Q3", "Q4_large"])
rc.groupby("bucket", observed=True)["population"].agg(["count", "mean", "median"]).round(0)


,count,mean,median
bucket,,,
Q1_small,65,3.807660e+05,165180.0
Q2,65,5.291082e+06,5081765.0
Q3,65,2.587411e+07,22462173.0
Q4_large,65,1.162502e+09,613861839.0


_Interpretation:_ across ~200 countries the mean population is much larger than the median - Q4 mean sits far above the others because China and India are outliers. Most countries are actually quite small.

## 3. SAS (.xpt) - NHANES demographics

In [ ]:
url = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/DEMO_L.xpt"
demo_full = pd.read_sas(url, format="xport")

demo = demo_full[["SEQN", "RIAGENDR", "RIDAGEYR", "RIDRETH3", "INDFMPIR"]].copy()
demo.columns = ["id", "gender", "age", "race", "poverty_ratio"]
demo["gender"] = demo["gender"].map({1: "Male", 2: "Female"})
demo["race"]   = demo["race"].map({1: "Mex-American", 2: "Other Hispanic",
                                   3: "NH-White",     4: "NH-Black",
                                   6: "NH-Asian",     7: "Other"})
demo = demo.sample(50, random_state=1).reset_index(drop=True)
demo.head()


,id,gender,age,race,poverty_ratio
0,142239.0,Female,68.0,Other,1.09
1,141054.0,Male,38.0,Mex-American,1.44
2,140785.0,Female,62.0,NH-White,3.01
3,137015.0,Female,80.0,NH-White,4.42
4,130774.0,Female,50.0,NH-White,0.23


In [ ]:
# Descriptive stats
demo[["age", "poverty_ratio"]].describe().round(2)


,age,poverty_ratio
count,50.00,40.00
mean,36.86,2.15
std,24.66,1.74
min,1.00,0.01
25%,14.25,0.82
50%,34.50,1.45
75%,61.25,3.68
max,80.00,5.00


In [ ]:
# Subset: adults (18+)
demo[demo["age"] >= 18].head()


,id,gender,age,race,poverty_ratio
0,142239.0,Female,68.0,Other,1.09
1,141054.0,Male,38.0,Mex-American,1.44
2,140785.0,Female,62.0,NH-White,3.01
3,137015.0,Female,80.0,NH-White,4.42
4,130774.0,Female,50.0,NH-White,0.23


In [ ]:
# Regex filter: non-Hispanic race categories (labels starting with "NH")
demo[demo["race"].fillna("").str.contains(r"^NH", regex=True)]["race"].value_counts()


race
NH-White    27
NH-Black     7
NH-Asian     4
Name: count, dtype: int64

In [ ]:
# Groupby/agg: mean age and poverty ratio by gender
demo.groupby("gender")[["age", "poverty_ratio"]].mean().round(2)


,age,poverty_ratio
gender,,
Female,43.79,2.02
Male,28.05,2.36


_Interpretation:_ in this 50-person random slice, mean age and poverty ratio differ between men and women. The `NH-*` groups (non-Hispanic white/Black/Asian) make up most of the sample, which matches NHANES's national coverage.

_Wrap-up:_ three formats, same simple pipeline - load, describe, slice, filter, group. Population data is heavy-tailed; a handful of large countries pull every mean upward. NHANES shows those same skews inside a single country's household income.